# PDB to ODE and NERDSS Workflow
## Tutorial 3 : benzaldehyde lyase mutant M6 from Herbiconiux sp. SALV-R1 (8y7s)

This tutorial demonstrates 8y7s as an example to use ProAffinity-GNN(1) to predict binding affinities. Note that for detailed explanation of the workflow, refer to tutorial 1: `ionerdss_tutorial_6bno.ipynb`

> ### Aims for this tutorial:
> - Give an example of a homo-tetramer assembly but has three different types of symmetric bindings
> - Tutorial on how to install and use ProAffinity-GNN to predict binding affinities

## Example: 8Y7S Structure

- In this file, we'll use **8Y7S** - HIV-1 CA-SP1 assembly as our example.

---
## Part 1: Preparation - Install Dependencies

ProAffinity-GNN(1) depends on PyTorch and AutoDockFR (ADFR)(2,3). The installation is specific to the hardware, conda/pip/wheel, the CUDA version, and the operating system. Therefore, we keep PyTorch and ADFR out of the default dependency files and the users need to install them separately.

### 1.1. Install PyTorch (v2.2.2)

Visit the following website and install based on your environment:

https://pytorch.org/get-started/previous-versions/#:~:text=org/whl/cpu-,v2.2.2,-Conda 

Required modules include:

```
torch
torchvision
torch-geometric
```

The ProAffinity-GNN is writtena and tested with v2.2.2, though other versions may also work.

> **WARNING:** PyTorch requires `numpy` version < 2.0.0. The lower version is installed by default with the provided conda environment. If you have a higher version of `numpy`, it is suggest to create a new conda environment with `numpy` version < 2.0.0 and install PyTorch again to avoid ABI compatibility issues.

### 1.2. Install ADFR

Visit the following website and install based on your environment:

https://ccsb.scripps.edu/adfr/downloads/

If you are using macOS, the macOS safety feature may prevent the installation. You can install with the provided install script `install_ADFR_mac.sh` (Only works for macOS). You might need to give it permission to run first:

```
chmod +x install_ADFR_mac.sh
```

Then run:

```
./install_ADFR_mac.sh
```



### 1.3. Install Optional Dependencies

```
pip install "ioNERDSS[proaffinity]"
```




---
## Part 2: Run the ionerdss pipeline 

Same as in the previous tutorial, we set up and run the ionerdss pipeline. Note that we turn on ProAffinity-GNN prediction by giving the `` flag.

> **WARNING!** Enabling ProAffinity-GNN can be heavily memory-consuming. 

In [ ]:
#  Path handling (standard library)
from pathlib import Path

# Core imports
import ionerdss as ion
from ionerdss import build_system_from_pdb

# For visualizations
import pandas as pd
import matplotlib.pyplot as plt

pdb_id = "8y7s"

# Build the system using simplified API
# This should take ~5 seconds for 8y7s
system = build_system_from_pdb(
    source=pdb_id,
    workspace_path=f"{pdb_id}_dir",
    interface_detect_distance_cutoff=0.6,
    interface_detect_n_residue_cutoff=3,
    chain_grouping_seq_threshold=0.5,

    # nerdss
    nerdss_total_molecule_count = 40,
    nerdss_n_itr = 100000,
    nerdss_water_box=[188, 188, 188], # = 10 uM

    # ProAffinity
    predict_affinity=True,  # Enable affinity prediction
    adfr_path='/Users/yueying/Documents/ADFR',  # Path to ADFR
    
    # ODE Pipeline Configuration
    ode_enabled=True,            # Now using System-compatible generator!
    ode_time_span=None,   # Auto-calculated based on NERDSS simulation time
    ode_solver_method="BDF",     # Solver for stiff systems
    ode_plot=True,               # Generate plots
    ode_save_csv=True            # Save data to CSV
)

# (Optional) List out generated files
# List all generated files
workspace_path = Path(f"{pdb_id}_dir")

print("Generated Files:")

print("\n NERDSS Input Files:")
nerdss_dir = workspace_path / "nerdss_files"
if nerdss_dir.exists():
    for file in sorted(nerdss_dir.glob("*.mol")) + sorted(nerdss_dir.glob("*.inp")):
        size = file.stat().st_size / 1024  # KB
        print(f"  nerdss_files/{file.name:<30} ({size:>6.1f} KB)")

print("\n System Data:")
outputs_dir = workspace_path / "outputs" / "systems"
if outputs_dir.exists():
    for file in sorted(outputs_dir.glob("*.json")):
        size = file.stat().st_size / 1024  # KB
        print(f"  outputs/systems/{file.name:<27} ({size:>6.1f} KB)")

print("\n System Builder Log:")
outputs_dir = workspace_path / "logs"
if outputs_dir.exists():
    for file in sorted(outputs_dir.glob("*.log")):
        size = file.stat().st_size / 1024  # KB
        print(f"  logs/{file.name:<38} ({size:>6.1f} KB)") 

---
## Part 2: Run NERDSS Simulation

### Run NERDSS with python subprocess

> **Note**: This requires NERDSS to be installed on your system. The user also has to specify the path to the NERDSS executable.

> **WARNING**: This may take a long time depending on your hardware (2 min - 20 min)!

In [ ]:
# run NERDSS with subprocess
import subprocess

# Check if NERDSS is available
# nerdss_cmd should be replaced with the actual path to the NERDSS executable
nerdss_cmd = "~/Workspace/Reaction_ode/nerdss_development/bin/nerdss"
nerdss_path = Path(nerdss_cmd).expanduser() # replaces tilde with appropriate user home path

if nerdss_path.exists():
    
    # Run NERDSS
    result = subprocess.run(
        f"{nerdss_cmd} -f parms.inp",
        shell=True,
        cwd=f"{pdb_id}_dir/nerdss_files",
        capture_output=True,
        text=True
    )
    
    if result.returncode == 0:
        print("✓ NERDSS simulation completed!")
        print(f"\nCheck {pdb_id}_dir/nerdss_files/ for output files")
    else:
        print("⚠ NERDSS simulation failed")
        print(result.stderr[:500])
else:
    print("⚠ NERDSS not found at:", nerdss_cmd)

---
## Part 3: Analyze NERDSS Output

After running NERDSS simulations, we can analyze the results using the `Analyzer` class.


In [ ]:
# Initialize Analyzer with NERDSS output directory
analysis = ion.Analyzer(f"{pdb_id}_dir")

# Display discovered simulations
print(f"Found {len(analysis.simulations)} simulation(s)")
for i, sim in enumerate(analysis.simulations):
    print(f"  [{i}] Simulation ID: {sim.id}")

#################
# Plot the NERDSS trajectory alone
plt.figure()

sim = analysis.get_simulation(0)
complex_compositions = [{"A":1},
                        {"A":2},
                        {"A":3},
                        {"A":4},
                        {"A":5},
                        {"A":10},
                        {"A":20}] 

# get the time series data for the above complexes
time, counts = sim.get_time_series(complex_compositions)

# plot all the returned data
for i in range(len(complex_compositions)):
    plt.plot(time,counts[i],label=str(complex_compositions[i]))

plt.legend()
plt.show()

In [ ]:
plt.figure()

average_size_time_series = sim.get_average_size_time_series()
plt.plot(average_size_time_series[0], average_size_time_series[1])


---
## (Optional) Part 4: Avoid kinetic trapping with titration

> Note: This step involves editing the `parms.inp` file after ioNERDSS generates the files. Although we showcase editing the file via python, direct editing or a .sh script might be easier depending on your task and your system. 

> **WARNINIG!** The NERDSS simulation in this step may take more than 48 hours to run (hardware dependent). Proceed with caution!

In [ ]:
from pathlib import Path
import re

def update_parms_inp(dir_path: str) -> None:
    p = Path(dir_path) / "parms.inp"
    text = p.read_text(encoding="utf-8")

    def set_key_value(block_start: str, block_end: str, key: str, value: str) -> None:
        nonlocal text
        # Capture the block so we only edit within it
        pattern = re.compile(
            rf"(?s)({re.escape(block_start)}\n)(.*?)(\n{re.escape(block_end)})"
        )
        m = pattern.search(text)
        if not m:
            raise ValueError(f"Could not find block: {block_start} ... {block_end}")

        head, body, tail = m.group(1), m.group(2), m.group(3)

        # Replace "key = ..." preserving indentation
        kv_pat = re.compile(rf"(?m)^(?P<indent>\s*){re.escape(key)}\s*=\s*.*$")
        if kv_pat.search(body):
            body = kv_pat.sub(lambda mm: f"{mm.group('indent')}{key} = {value}", body, count=1)
        else:
            # If key doesn't exist, append it with 4-space indent (matches your file style)
            body = body.rstrip("\n") + f"\n    {key} = {value}\n"

        text = text[:m.start()] + head + body + tail + text[m.end():]

    # --- 1) parameters ---
    set_key_value("start parameters", "end parameters", "nItr", "10000000.0")
    set_key_value("start parameters", "end parameters", "overlapSepLimit", "2.3")

    # --- 2) boundaries ---
    # Keep your inline comment if present; replace the whole WaterBox assignment line.
    def set_waterbox(value: str) -> None:
        nonlocal text
        pattern = re.compile(r"(?m)^(?P<indent>\s*)WaterBox\s*=\s*\[.*?\](?P<comment>\s*#.*)?\s*$")
        m = pattern.search(text)
        if not m:
            raise ValueError("Could not find WaterBox line")
        indent = m.group("indent")
        comment = m.group("comment") or ""
        text = pattern.sub(f"{indent}WaterBox = {value}{comment}".rstrip(), text, count=1)

    set_waterbox("[250, 250, 250]")

    # --- 3) molecules ---
    # Set A : 50 within the "start molecules" block, preserving indentation.
    mol_block_pat = re.compile(r"(?s)(start molecules\n)(.*?)(\nend molecules)")
    m = mol_block_pat.search(text)
    if not m:
        raise ValueError("Could not find molecules block")

    head, body, tail = m.group(1), m.group(2), m.group(3)
    a_line_pat = re.compile(r"(?m)^(?P<indent>\s*)A\s*:\s*\d+(\s*)$")
    if a_line_pat.search(body):
        body = a_line_pat.sub(lambda mm: f"{mm.group('indent')}A : 50", body, count=1)
    else:
        body = body.rstrip("\n") + "\n    A : 50\n"

    text = text[:m.start()] + head + body + tail + text[m.end():]

    # --- 4) reactions: insert titration block right after 'start reactions' ---
    titration_block = (
        "\n"
        "    # Titration\n"
        "    0 -> A(aa1f, aa1b, aa2, aa3f, aa3b)\n"
        "    onRate3Dka = 0.00005 # M/s\n"
        "\n"
    )

    # Don’t double-insert if it already exists
    if re.search(r"(?m)^\s*#\s*Titration\s*$", text) is None:
        sr_pat = re.compile(r"(?m)^(start reactions\s*)$")
        m = sr_pat.search(text)
        if not m:
            raise ValueError("Could not find 'start reactions' line")
        insert_at = m.end()
        text = text[:insert_at] + titration_block + text[insert_at:]

    p.write_text(text, encoding="utf-8")
    print(f"Updated: {p}")
    print("\n===== parms.inp =====\n")
    print(text)

# Run the defined function
update_parms_inp(f"{pdb_id}_dir/nerdss_files")


Due to the time requirement for running this simulation with the edited `parms.inp` (~48 hours), we do not suggest the user start the simulation with a python subprocess. Instead, we recommend the user to run in background and redirect output to a .log file via a command that survives log-outs.


```
nohup ./<nerdss_directory>/nerdss -f parms.inp > nerdss_output.log 2>&1 &
```

Alternatively, the user can use multiplexers such as `tmux` or `screen` (GNU). Remember to include the command (without `nohup`) in the `sbatch` script if the user is on a `slurm`-managed supercomputer cluster. 

---
## Summary and Next Steps

### What We've Done

 Loaded 5L93 structure  
 Align the structures on a sphere  
 Exported NERDSS simulation files  
 Ran NERDSS simulation for the assembly of the sphere extending beyond the PDB file  
 Configured NERDSS simulation parameters for titration  

---

## Reference

(1) ProAffinity-GNN: A Novel Approach to Structure-Based Protein–Protein Binding Affinity Prediction via a Curated Data Set and Graph Neural Networks
Zhiyuan Zhou, Yueming Yin, Hao Han, Yiping Jia, Jun Hong Koh, Adams Wai-Kin Kong, and Yuguang Mu Journal of Chemical Information and Modeling 2024 64 (23), 8796-8808
DOI: 10.1021/acs.jcim.4c01850

(2) Pradeep Anand Ravindranath, Stefano Forli, David S. Goodsell, Arthur J. Olson, Michel F. Sanner (2015) AutoDockFR: Advances in Protein-Ligand Docking with Explicitly Specified Binding Site Flexibility. PLOS Computational Biology 11(12): e1004586. https://doi.org/10.1371/journal.pcbi.1004586

(3) Yong Zhao Daniel Stoffler Michel Sanner (2006) Hierarchical and multi-resolution representation of protein flexibility. Bioinformatics, Volume 22, Issue 22, 15 November 2006, Pages 2768–2774, https://doi.org/10.1093/bioinformatics/btl481

---

*Tutorial created: 2026-2-4*  
*IONERDSS Version: 1.2.0*